<a href="https://colab.research.google.com/github/bsheese/387/blob/main/19_onramp/19_1_ML_Foundations/19_1_4_Softmax_and_the_Loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The On-Ramp: Part 4
## Softmax and the Loss

---

## What This Notebook Is About

In 19_1_3 we saw that the Pets model's last layer computes four scores, one weighted sum per breed. This notebook finishes the path from those scores to the model's prediction, and then looks at how training measures how wrong a prediction is.

1. **Softmax**: how four scores, which can be any numbers, become four probabilities that add up to 1
2. **The loss**: how training measures how wrong a probability is, using the logarithm

**Optional reference.** If you took CS 377, you have seen these ideas before in [18_5_1](https://github.com/bsheese/cs377/blob/main/18_classification/18_5_MutliClassClassification/18_5_1_Multiclass_Intro.ipynb). If you haven't, that notebook covers the same ideas on another dataset, in more depth. You don't need it for this course.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    # It saves to a temporary name first, so a download that fails partway doesn't leave a broken file behind.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name + '.part')
        os.replace(name + '.part', name)
    return name

preds = pd.read_csv(fetch('pets4_predictions.csv'))
BREEDS = ['Birman', 'Ragdoll', 'basset_hound', 'beagle']
scores = preds[[f'score_{b}' for b in BREEDS]].to_numpy()
probs = preds[[f'prob_{b}' for b in BREEDS]].to_numpy()
print('scores:', scores.shape, '  probs:', probs.shape)

---

## Section 1: From Scores to Probabilities: Softmax

Here are the four scores and four probabilities that 19_0_2 saved for the first photo, a Birman:

In [ ]:
print('photo:', preds.loc[0, 'file'])
print('scores:       ', scores[0].round(3))
print('probabilities:', probs[0].round(3))

The scores can't be used as probabilities directly. Some are negative, and they don't add up to 1. The function that turns them into probabilities is called **softmax**, and it has two steps:

1. **Make every score positive** by raising $e$ (2.718...) to the power of the score. `np.exp` does this. The bigger the score, the bigger the result, so the order of the breeds doesn't change.
2. **Divide each result by the total**, so the four numbers add up to 1.

Here is how $e^{x}$ behaves for a few values of $x$. It's always positive, it's exactly 1 at $x = 0$, and it grows very quickly as $x$ increases:

In [ ]:
for value in [-4, -2, -1, 0, 1, 2, 4]:
    print(f'e^{value:<2} = {np.exp(value):8.3f}')

Now the two steps on the first photo's scores:

In [ ]:
s = scores[0]
step1 = np.exp(s)
print('step 1, e^score:     ', step1.round(3))
print('total:               ', step1.sum().round(3))
step2 = step1 / step1.sum()
print('step 2, divided:     ', step2.round(3))
print('saved probabilities: ', probs[0].round(3))

They match, so softmax is exactly what the model does to its scores.

To apply softmax to all 800 photos at once, each row has to be divided by its own total. `.sum(axis=1)` adds across each row, and `keepdims=True` keeps the result as a column of shape (800, 1), so that dividing the (800, 4) array by it divides every row by its own total. (This is the broadcasting from 19_1_3.)

In [ ]:
def softmax(scores):
    exps = np.exp(scores)
    return exps / exps.sum(axis=1, keepdims=True)

print('row totals:', np.exp(scores).sum(axis=1, keepdims=True).shape)
mine = softmax(scores)
print('largest difference from the saved probabilities:', np.abs(mine - probs).max().round(6))
print('every row adds up to 1:', np.allclose(mine.sum(axis=1), 1))

### What softmax pays attention to

Softmax only depends on the **differences** between the scores. Adding the same number to all four scores multiplies every $e^{\text{score}}$ by the same amount, which cancels when we divide by the total. But making the differences bigger makes the model more confident:

In [ ]:
s = np.array([[2.0, 1.0, 0.0, -1.0]])
print('scores             ', s[0], ' -> ', softmax(s)[0].round(3))
print('scores + 10        ', (s + 10)[0], ' -> ', softmax(s + 10)[0].round(3))
print('scores x 3         ', (s * 3)[0], ' -> ', softmax(s * 3)[0].round(3))
print('scores x 0.1       ', (s * 0.1)[0], ' -> ', softmax(s * 0.1)[0].round(3))

Multiplying the scores by 3 spreads them apart, and the top breed's probability goes from 0.64 to 0.95. Multiplying by 0.1 squeezes them together, and the probabilities move toward 0.25 each. Unit 23 uses this to control how adventurous a text generator is.

---

## Section 2: How Wrong Is a Probability? The Loss

19_1_1 plotted the **loss** during training and said only that lower is better. Now we can say what it is. For each photo, take the probability the model gave to the **true** breed. If that probability is close to 1, the model did well; if it's close to 0, the model was confidently wrong. The loss turns this into a penalty with the **natural logarithm**, `np.log`:

$$\text{loss for one photo} = -\log(\text{probability of the true breed})$$

The logarithm undoes $e^{x}$: $\log(e^{x}) = x$. So $\log(1) = 0$, and the log of a number between 0 and 1 is negative, more and more so as the number gets closer to 0. The minus sign makes the penalty positive.

In [ ]:
print('probability of     penalty')
print('the true breed     -log(p)')
for p in [0.99, 0.9, 0.5, 0.25, 0.1, 0.01]:
    print(f'{p:12}       {-np.log(p):6.2f}')

A right answer given with high confidence costs almost nothing, a 50/50 answer costs 0.69, and a confident wrong answer (0.01 for the true breed) costs 4.6. Being confidently wrong is penalized far more than being unsure. This penalty is called the **cross-entropy loss**, and the loss of a whole set of photos is the average of the photos' penalties. It's what `nn.CrossEntropyLoss` computed in 19_0_1 and 19_1_1.

Here it is for the first five test photos, one photo at a time. For each one, look up its true breed, take the probability in that breed's column, and compute the penalty.

In [ ]:
is_test = preds['split'].to_numpy() == 'test'
test_rows = np.where(is_test)[0]          # positions of the test photos in preds

for row in test_rows[:5]:
    true_breed = preds.loc[row, 'breed']
    p = probs[row, BREEDS.index(true_breed)]          # the probability given to the true breed
    print(f"{preds.loc[row, 'file']:15} true breed {true_breed:8} probability {p:.3f}   penalty {-np.log(p):.2f}")

For a whole set of photos, the same lookup is done in one step. `probs[rows, cols]`, with an array of row positions and an array of column positions, picks out one element per row: here, for each test photo, the probability in the column of its true breed. The first five values match the loop above.

In [ ]:
is_test = preds['split'].to_numpy() == 'test'
true_col = np.array([BREEDS.index(b) for b in preds['breed']])

p_true = probs[is_test, :][np.arange(is_test.sum()), true_col[is_test]]
print('probability of the true breed, first five test photos:', p_true[:5].round(3))
print(f'average loss on the test photos: {(-np.log(p_true)).mean():.3f}')
print(f'worst single photo: p = {p_true.min():.3f}, penalty {-np.log(p_true.min()):.2f}')

The average penalty over the 200 test photos is 0.286. Most photos cost very little, like the five above, and the worst single photo, where the model gave the true breed a probability of only 0.047, costs 3.06 by itself. Training a network means changing its weights to make this average smaller. Unit 20 shows how.

---

## Summary

- **Softmax** turns a list of scores into probabilities: raise $e$ to the power of each score (making them positive), then divide by the total (making them add up to 1). It reproduces the Pets model's saved probabilities exactly. Only the differences between scores matter, and bigger differences give more confident probabilities.
- The **cross-entropy loss** of a photo is $-\log$ of the probability given to its true breed. It's near 0 for confident right answers and very large for confident wrong ones. Training makes the average loss smaller.

## What's next

Softmax and the loss complete the Pets model's last layer: four weighted sums of the 512 features, softmax to turn them into probabilities, and a loss that training makes smaller. 19_1_5 asks what a weighted sum can't do. It uses XOR, a problem with four points that no single neuron can solve, and shows that putting a layer of neurons in front of the last one solves it. That is the reason every network in this course, including the Pets model, has more than one layer.